# 1 - Simulación y Generación de Datos Sintéticos (FinanceAI)

**1. Simulación** > 2. EDA > 3. Entrenamiento

Este cuaderno genera un conjunto de datos sintéticos autónomo y limpio. Construye la semilla poblacional y transaccional con la que inicializaremos la base de datos de MySQL y sobre la cual entrenaremos nuestros modelos.

**Objetivos:**
- Generar 3000 usuarios con perfiles financieros calculados lógicamente (con un margen de ruido).
- Generar 15000 transacciones para las 10 categorías oficiales estipuladas.
- Particionar los datos de origen en **Train (60%)**, **Val (20%)** y **Test (20%)** para aislar el proceso y prevenir Data Leakage absoluto en los modelos.

In [1]:
import pandas as pd
import numpy as np
import os
import json

# Fijar semilla de aleatoriedad para garantizar reproducibilidad total en el proyecto
SEED = 42
np.random.seed(SEED)

## 1. Simulación de Usuarios (Perfilado)
Construiremos a los usuarios basando su salud financiera en la relación entre el nivel de deuda y su hábito de ahorro.

In [2]:
n_usuarios = 3000

nombres = ['Ana', 'Juan', 'Sofia', 'Pedro', 'Laura', 'Diego', 'Valentina', 'Carlos', 'Camila', 'Luis',
           'Maria', 'Jorge', 'Lucia', 'Miguel', 'Marta', 'Alejandro', 'Elena', 'Martin', 'Clara', 'Andres']

# Generación de ingresos (sueldos entre 500 y 6000)
ingresos = np.round(np.random.uniform(500, 6000, size=n_usuarios), 2)

# Nivel de endeudamiento base (0% a 90% del ingreso)
endeudamientos = np.round(np.random.uniform(0, 90, size=n_usuarios), 2)

# Frecuencia de ahorro
opciones_ahorro = ['Alta', 'Media', 'Baja', 'Ninguna']
prob_ahorro = [0.2, 0.4, 0.25, 0.15] 
frecuencias_ahorro = np.random.choice(opciones_ahorro, size=n_usuarios, p=prob_ahorro)

perfiles = []
for i in range(n_usuarios):
    end = endeudamientos[i]
    ahorro = frecuencias_ahorro[i]
    
    # Reglas lógicas de negocio
    if end > 50.0:
        perf = 'En riesgo'
    elif end > 35.0 or ahorro == 'Ninguna':
        perf = 'En observacion'
    elif end <= 35.0 and ahorro in ['Media', 'Alta']:
        perf = 'Saludable'
    else:
        perf = 'En observacion'
        
    # Inyección de ruido (15%) para evitar precisión algorítmica perfecta (evitar overfitting en ML)
    if np.random.rand() < 0.15:
        perf = np.random.choice(['Saludable', 'En observacion', 'En riesgo'])
        
    perfiles.append(perf)

# Conformación del DataFrame
df_usuarios = pd.DataFrame({
    'id': range(1, n_usuarios + 1),
    'nombre': np.random.choice(nombres, size=n_usuarios),
    'ingreso_mensual': ingresos,
    'nivel_endeudamiento': endeudamientos,
    'frecuencia_ahorro': frecuencias_ahorro,
    'perfil_financiero': perfiles
})

print("Usuarios simulados:", df_usuarios.shape)
print(df_usuarios['perfil_financiero'].value_counts(normalize=True) * 100)

Usuarios simulados: (3000, 6)
perfil_financiero
En riesgo         43.5
En observacion    31.0
Saludable         25.5
Name: proportion, dtype: float64


## 2. Simulación de Transacciones (Clasificador NLP)
Generaremos 15000 consumos y los mapearemos a las 10 categorías definitivas establecidas para el proyecto.

In [3]:
n_transacciones = 15000

# Las 10 categorías requeridas (sin acentos)
diccionario_conceptos = {
    'Alimentacion': ['supermercado coto', 'verduleria el sol', 'carniceria central', 'almacen san martin', 'compra panaderia', 'supermercado carrefour', 'compras fiambreria', 'compra dia'],
    'Educacion': ['cuota universidad', 'compra libros', 'curso de programacion', 'matricula colegio', 'utiles escolares', 'taller ingles', 'cuota jardin'],
    'Electrodomesticos': ['compra heladera', 'lavarropas fravega', 'televisor garbarino', 'microondas musimundo', 'licuadora philips', 'pava electrica', 'aire acondicionado'],
    'Inversion': ['compra dolares', 'fondo comun inversion', 'plazo fijo', 'cedears', 'bonos del estado', 'acciones ypf', 'transferencia broker'],
    'Ocio': ['salida cine', 'suscripcion netflix', 'cena restaurante', 'entradas recital', 'cerveceria', 'suscripcion spotify', 'juegos steam', 'cafeteria'],
    'Salud': ['estudios clinicos', 'compra farmacia', 'consulta medica', 'cuota prepaga', 'medicamentos', 'dentista', 'analisis sangre', 'optica'],
    'Servicios': ['factura luz edesur', 'abono internet', 'servicio agua', 'factura gas', 'telefonia movil', 'impuesto municipal', 'abl', 'rentas'],
    'Transporte': ['viaje uber', 'colectivo', 'carga sube', 'combustible ypf', 'peaje autopista', 'taxi', 'viaje cabify', 'combustible shell'],
    'Vestimenta': ['compra zapatillas', 'pantalon jean', 'remera algodon', 'campera invierno', 'ropa deportiva', 'local indumentaria', 'zapatos'],
    'Vivienda': ['pago alquiler', 'expensas edificio', 'servicio plomeria', 'ferreteria', 'pintura habitacion', 'reparacion electrica', 'materiales construccion']
}

categorias = list(diccionario_conceptos.keys())
usuario_ids = np.random.randint(1, n_usuarios + 1, size=n_transacciones)
selected_categories = np.random.choice(categorias, size=n_transacciones)

# Fechas a lo largo del 2023
fechas_random = pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, size=n_transacciones), unit='d')

descripciones = []
montos = []

for cat in selected_categories:
    desc = np.random.choice(diccionario_conceptos[cat])
    
    # Inyección de ruido en texto (simulando errores de usuario o posnet)
    if np.random.rand() < 0.20:
        prefijos = ["pago ", "compra ", "tarjeta ", "fac ", ""]
        desc = str(np.random.choice(prefijos)) + desc
    if np.random.rand() < 0.10:
        desc = desc.replace("a", "q", 1) if "a" in desc else desc.replace("e", "w", 1)

    monto = round(np.random.uniform(10, 1500), 2)
    descripciones.append(desc)
    montos.append(monto)

# Inyección de ruido en categorías (10%) para evitar que el NLP sea 100% perfecto
ruido_mask = np.random.rand(n_transacciones) < 0.10
categorias_ruido = np.random.choice(categorias, size=ruido_mask.sum())
selected_categories[ruido_mask] = categorias_ruido

df_transacciones = pd.DataFrame({
    'id': range(1, n_transacciones + 1),
    'usuario_id': usuario_ids,
    'descripcion': descripciones,
    'valor': montos,
    'categoria': selected_categories,
    'fecha': fechas_random.strftime('%Y-%m-%d')
})

print("Transacciones simuladas:", df_transacciones.shape)
print(df_transacciones['categoria'].value_counts())
print("\nMuestra del conjunto:")
print(df_transacciones.head())

Transacciones simuladas: (15000, 6)
categoria
Vivienda             1588
Salud                1564
Servicios            1554
Inversion            1548
Vestimenta           1485
Transporte           1480
Educacion            1464
Electrodomesticos    1448
Ocio                 1447
Alimentacion         1422
Name: count, dtype: int64

Muestra del conjunto:
   id  usuario_id          descripcion    valor     categoria       fecha
0   1        2808       ropa deportiva   195.51    Vestimenta  2023-01-20
1   2        1196           plazo fijo  1487.47     Inversion  2023-04-11
2   3        1754           comprq dia   152.11  Alimentacion  2023-10-22
3   4        1673  pqgo remera algodon    73.71    Vestimenta  2023-06-06
4   5        2369         juegos steam    47.54          Ocio  2023-01-01


## 3. Prevención de Data Leakage (Partición de Datos)
Antes de exportar, asignamos un identificador de partición a cada usuario para asegurar que las transacciones y perfiles de validación y test sean totalmente opacos para el análisis exploratorio (EDA) y el entrenamiento de los algoritmos.

In [4]:
# Distribución de los particiones: 60% Train, 20% Validación, 20% Test
splits = np.random.choice(['train', 'val', 'test'], size=n_usuarios, p=[0.6, 0.2, 0.2])
df_usuarios['split'] = splits

# Mapeamos el split del usuario hacia sus transacciones para garantizar coherencia estructural
split_map = dict(zip(df_usuarios['id'], df_usuarios['split']))
df_transacciones['split'] = df_transacciones['usuario_id'].map(split_map)

print("Distribución de Usuarios por Partición:")
print(df_usuarios['split'].value_counts(normalize=True) * 100)
print("\nDistribución de Transacciones por Partición:")
print(df_transacciones['split'].value_counts(normalize=True) * 100)

Distribución de Usuarios por Partición:
split
train    59.300000
test     20.533333
val      20.166667
Name: proportion, dtype: float64

Distribución de Transacciones por Partición:
split
train    59.306667
test     20.500000
val      20.193333
Name: proportion, dtype: float64


## 4. Exportación
Exportaremos las tablas a formato CSV para la ingesta en Pandas en los próximos cuadernos, y en JSON como semilla de inserción para el Backend en Java.

In [5]:
os.makedirs('data', exist_ok=True)

# Guardado CSV
df_usuarios.to_csv('data/usuarios.csv', index=False)
df_transacciones.to_csv('data/transacciones.csv', index=False)

# Guardado JSON (asegurando utf-8 sin conversión unicode)
df_usuarios.to_json('data/usuarios.json', orient='records', indent=2, force_ascii=False)
df_transacciones.to_json('data/transacciones.json', orient='records', indent=2, force_ascii=False)

print("¡Exportación exitosa a la carpeta /data/!")

¡Exportación exitosa a la carpeta /data/!
